# Исследование набора дневной истории 

- Выгрузка исторических данных по нескольким инструментам
- Выгрузка свечей за период с приминением пагинации
- Анализ доступности данных и интервалов



In [1]:
import pandas as pd
import requests

pd.set_option('display.max_columns', None)

In [2]:
ENGINE = 'stock'
MARKET = 'shares'
BOARD = 'TQBR'
SECURITES = ["SBER", "GAZP", "LKOH",]

In [4]:
# 1.загрузить дневную историю для нескольких инструментов
params = {
    "from": "2026-07-01",
    "till": "2026-07-31",
    "lang": "ru"
}

def get_daily_history_frames(url:str, params: dict, company: str)-> pd.DataFrame:
    respons = requests.get(
        url=url,
        params=params,
        timeout=30,
    )
    print(company)
    print(respons.url)
    print(respons.status_code)

    data = respons.json()
    df = pd.DataFrame(
        columns=data['history']['columns'],
        data=data['history']['data']
    )
    return df

daily_history_frames = []
for company in  SECURITES:
    url = f'https://iss.moex.com/iss/history/engines/{ENGINE}/markets/{MARKET}/boards/{BOARD}/securities/{company}.json'
    daily_history_frames.append(get_daily_history_frames(url, params, company))

daily_history_df  = pd.concat(daily_history_frames)
daily_history_df.head(5)

SBER
https://iss.moex.com/iss/history/engines/stock/markets/shares/boards/TQBR/securities/SBER.json?from=2026-07-01&till=2026-07-31&lang=ru
200
GAZP
https://iss.moex.com/iss/history/engines/stock/markets/shares/boards/TQBR/securities/GAZP.json?from=2026-07-01&till=2026-07-31&lang=ru
200
LKOH
https://iss.moex.com/iss/history/engines/stock/markets/shares/boards/TQBR/securities/LKOH.json?from=2026-07-01&till=2026-07-31&lang=ru
200


,BOARDID,TRADEDATE,SHORTNAME,SECID,NUMTRADES,VALUE,OPEN,LOW,HIGH,LEGALCLOSEPRICE,WAPRICE,CLOSE,VOLUME,MARKETPRICE2,MARKETPRICE3,ADMITTEDQUOTE,MP2VALTRD,MARKETPRICE3TRADESVALUE,ADMITTEDVALUE,WAVAL,TRADINGSESSION,CURRENCYID,TRENDCLSPR,TRADE_SESSION_DATE
0,TQBR,2026-07-01,Сбербанк,SBER,181706,1.257712e+10,308.12,305.55,311.57,306.68,308.41,307.04,40780087,308.48,308.48,None,1.066855e+10,1.066855e+10,None,0,3,SUR,-0.34,2026-07-01
1,TQBR,2026-07-02,Сбербанк,SBER,197020,1.393199e+10,307.10,299.83,308.20,301.80,303.31,301.05,45930395,303.30,303.30,None,1.229356e+10,1.229356e+10,None,0,3,SUR,-1.95,2026-07-02
2,TQBR,2026-07-03,Сбербанк,SBER,161463,9.917346e+09,301.65,298.04,303.25,301.75,300.92,301.19,33006616,300.32,300.32,None,8.649960e+09,8.649960e+09,None,0,3,SUR,0.05,2026-07-03
3,TQBR,2026-07-06,Сбербанк,SBER,323901,1.508306e+10,301.39,294.07,305.60,298.68,300.40,298.77,50214521,300.12,300.12,None,1.187651e+10,1.187651e+10,None,0,3,SUR,-0.80,2026-07-06
4,TQBR,2026-07-07,Сбербанк,SBER,248284,1.459397e+10,299.02,292.12,299.47,296.93,295.40,298.98,49403734,295.01,295.01,None,1.250731e+10,1.250731e+10,None,0,3,SUR,0.07,2026-07-07


In [ ]:
# 2.Загрузить минутные свечи для несколькизх иснтрументов дял отрабоки пагинации
params_candles = {
    "from": "2026-07-01",
    "till": "2026-07-05",
    "interval": 1,
}

def get_candles_page(url:str, params: dict, company_tiket: str)-> pd.DataFrame:
    response = requests.get(
        url=url,
        params=params,
        timeout=30,
    )

    if response.status_code != 200:
        print(f'Ошибка выгрузки, код: {response.status_code}')

    data = response.json()

    df = pd.DataFrame(
        columns=data['candles']['columns'],
        data=data['candles']['data']
    )
    df['secid'] = company_tiket
    return df

candles_frames = []

for security in SECURITES:
    url = f'https://iss.moex.com/iss/engines/{ENGINE}/markets/{MARKET}/boards/{BOARD}/securities/{security}/candles.json'
    params_candles["start"] = 0

    while True:
        df = get_candles_page(url, params_candles, security)

        if df.empty:
            break
        else:
            params_candles['start'] += df.shape[0]
            candles_frames.append(df)

df_candles = pd.concat(candles_frames, ignore_index=True)
df_candles.shape[0]

12297

In [ ]:
# Получение периода доступных свечей и справочника интервалов доступных свечей по инструменту SBER
url_candleborders = f'https://iss.moex.com//iss/engines/{ENGINE}/markets/{MARKET}/boards/{BOARD}/securities/SBER/candleborders.json'
response_candleborders = requests.get(
        url=url_candleborders,
        timeout=30,
    )

data_candleborders = response_candleborders.json()

df_borders = pd.DataFrame(
    columns= data_candleborders['borders']['columns'],
    data= data_candleborders['borders']['data']
)

df_durations = pd.DataFrame(
    columns= data_candleborders['durations']['columns'],
    data= data_candleborders['durations']['data']
)

In [8]:
df_borders.head()

,begin,end,interval
0,2011-12-15 10:00:00,2026-08-06 07:59:59,1
1,2007-07-01 00:00:00,2026-09-30 00:00:00,4
2,2007-07-16 00:00:00,2026-08-09 00:00:00,7
3,2011-12-08 10:00:00,2026-08-06 07:15:01,10
4,2007-07-20 00:00:00,2026-08-06 07:14:56,24


In [9]:
df_durations.head()

,interval,duration,days,title,hint
0,1,60,None,минута,1м
1,10,600,None,10 минут,10м
2,60,3600,None,час,1ч
3,24,86400,None,день,1д
4,7,604800,None,неделя,1н
